In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import boxcox

from sklearn.preprocessing import StandardScaler,MinMaxScaler
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error

from sklearn.preprocessing import PolynomialFeatures
from itertools import combinations

from sklearn.preprocessing import RobustScaler

import xgboost as xgb
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from itertools import product
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge, Lasso, BayesianRidge
from sklearn.svm import SVR
from sklearn.pipeline import make_pipeline
import joblib

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from scipy.stats import loguniform, randint, uniform

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, PReLU
from tensorflow.keras.optimizers import Adam, SGD, RMSprop, AdamW
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
data=pd.read_excel("data - copie.xlsx")
data = data.replace({',': '.'}, regex=True)
data.columns=['p','v','h','t','Ra','Desnity','VED','LED','PED']
X_selected=['p','v','h','t']


In [3]:

data2=pd.read_excel('synthetic_data_gmm_1400.xlsx')
data2 = data2.replace({',': '.'}, regex=True)

data2.columns=['p','v','h','t','Ra']


In [6]:

data3 = pd.read_csv("synthetic_data_gans.csv", delimiter=',')
data3 = data3.replace({',': '.'}, regex=True)
data3.columns=['p','v','h','t','Ra']


In [7]:

df1 = data[['p', 'v', 'h', 't', 'Ra']]
df2 = data2[['p', 'v', 'h', 't', 'Ra']]
df3 = data3[['p', 'v', 'h', 't', 'Ra']]


merged_data = pd.concat([df1, df2, df3], ignore_index=True)
print(merged_data.shape)  

X_merged = merged_data[X_selected].values
y_merged = merged_data[['Ra']].values


(6536, 5)


In [8]:
def box_cox_fun(data, lambda_val=0.5):
    return ((data + 1e-5) ** lambda_val - 1) / lambda_val

def inverse_box_cox(data, lambda_val=0.5):
    if lambda_val == 0:
        return np.exp(data)
    else:
        return (data * lambda_val + 1) ** (1 / lambda_val) - 1e-5


In [ ]:
scaler=RobustScaler()
X_scaled_combined =scaler.fit_transform(X_merged)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_combined,y_merged , 
    test_size=0.30, 
    random_state=42,shuffle=True)
y_train_bc=box_cox_fun(y_train)
y_test_bc=box_cox_fun(y_test)

In [10]:





def get_optimizer(optimizer_name, learning_rate):
    optimizers_dict = {
        'adam': Adam(learning_rate=learning_rate),
        'adamw': AdamW(learning_rate=learning_rate, weight_decay=1e-4),
        'sgd': SGD(learning_rate=learning_rate),
        'rmsprop': RMSprop(learning_rate=learning_rate),
    }
    
    if optimizer_name.lower() not in optimizers_dict:
        raise ValueError(f"Optimiseur '{optimizer_name}' non supporté ! Choisis parmi {list(optimizers_dict.keys())}")
    
    return optimizers_dict[optimizer_name.lower()]

def build_model(input_dim, optimizer_name, learning_rate):
    opt = get_optimizer(optimizer_name, learning_rate)
    
    model = Sequential([
        Dense(512, input_dim=input_dim),
        BatchNormalization(),
        PReLU(),

        Dense(256),
        BatchNormalization(),
        PReLU(),
        Dropout(0.3),

        Dense(128),
        BatchNormalization(),
        PReLU(),
        Dropout(0.3),

        Dense(64),
        BatchNormalization(),
        PReLU(),
        Dropout(0.3),

        Dense(32),
        BatchNormalization(),
        PReLU(),
        Dropout(0.3),

        Dense(16),
        BatchNormalization(),
        PReLU(),
        Dropout(0.2),

        Dense(1, activation='linear')
    ])
    
    model.compile(optimizer=opt, loss=tf.keras.losses.Huber(delta=1.0), metrics=['mae'])
    return model


early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=70,
    restore_best_weights=True
)


optimizers = ['adam', 'adamw', 'rmsprop']
learning_rates = [0.001, 0.005]
batch_sizes = [16,32,64,128]


best_score = float('inf')
best_params = {}
best_model = None
results = []


for opt_name in optimizers:
    for lr in learning_rates:
        for bs in batch_sizes:
            print(f"\n🔵 Test: optimizer={opt_name}, learning_rate={lr}, batch_size={bs}")
            
            model = build_model(X_train.shape[1], opt_name, lr)
            
            history = model.fit(
                X_train, y_train_bc,
                epochs=5000,
                batch_size=bs,
                validation_data=(X_test, y_test_bc),
                verbose=0,
                callbacks=[early_stopping]
            )
            
            preds = model.predict(X_test).squeeze()
            preds = inverse_box_cox(preds)
            
            rmse = np.sqrt(mean_squared_error(y_test, preds))
            mae = mean_absolute_error(y_test, preds)
            r2 = r2_score(y_test, preds)
            
           
            results.append({
                'optimizer': opt_name,
                'learning_rate': lr,
                'batch_size': bs,
                'RMSE': rmse,
                'MAE': mae,
                'R2': r2
            })
            
           
            if rmse < best_score:
                best_score = rmse
                best_params = {'optimizer': opt_name, 'learning_rate': lr, 'batch_size': bs}
                best_model = model


results_df = pd.DataFrame(results)
print("\n📊 Résumé de tous les tests :")
print(results_df.sort_values('RMSE'))


print("\n✅ Meilleurs hyperparamètres trouvés :")
print(best_params)
print(f"RMSE: {best_score:.4f}")


y_pred_final = best_model.predict(X_test)
y_pred_final = inverse_box_cox(y_pred_final)
r2_rn=r2_score(y_test, y_pred_final)
mae_rn=mean_absolute_error(y_test, y_pred_final)
rmse_rn=np.sqrt(mean_squared_error(y_test, y_pred_final))
print("\n📈 Résultats finaux sur le test set :")
print(f"R2 : {r2_rn:.4f}")
print(f"MAE : {mae_rn:.4f}")
print(f"RMSE : {rmse_rn:.4f}")



🔵 Test: optimizer=adam, learning_rate=0.001, batch_size=16
41/41 [==============================] - 1s 5ms/step

🔵 Test: optimizer=adam, learning_rate=0.001, batch_size=32
41/41 [==============================] - 0s 3ms/step

🔵 Test: optimizer=adam, learning_rate=0.001, batch_size=64
41/41 [==============================] - 1s 5ms/step

🔵 Test: optimizer=adam, learning_rate=0.001, batch_size=128
41/41 [==============================] - 1s 6ms/step

🔵 Test: optimizer=adam, learning_rate=0.005, batch_size=16
41/41 [==============================] - 0s 3ms/step

🔵 Test: optimizer=adam, learning_rate=0.005, batch_size=32
41/41 [==============================] - 0s 3ms/step

🔵 Test: optimizer=adam, learning_rate=0.005, batch_size=64
41/41 [==============================] - 1s 7ms/step

🔵 Test: optimizer=adam, learning_rate=0.005, batch_size=128
41/41 [==============================] - 1s 7ms/step

🔵 Test: optimizer=adamw, learning_rate=0.001, batch_size=16
41/41 [==========================

In [16]:
best_model.save("models_pvht_gmm_gan/best_model.h5")